# Imports

In [1]:
import json
import random
import h5py
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool, cpu_count

import pyroomacoustics as pra
from pyroomacoustics.directivities import DirectionVector, CardioidFamily
from scipy.spatial import ConvexHull

# Config

In [2]:
tCFG = {
    "num_rirs": 50000,
    "sample_rate": 32000,

    "rir_max_length_sec": 3.0,

    "room": {
        "types": ["shoebox", "random"],
        "min_dim": [3.0, 3.0, 2.5],
        "max_dim": [10.0, 8.0, 4.0],
        "rt60_range": [0.2, 0.8],
        "num_sources": 6,
    },

    "materials": {
        "brick":  [0.03, 0.04, 0.05, 0.07, 0.08],
        "curtain": [0.2, 0.35, 0.55, 0.7, 0.7],
        "wood":   [0.1, 0.15, 0.2, 0.25, 0.3],
        "carpet": [0.15, 0.25, 0.4, 0.6, 0.65],
        "concrete":[0.01, 0.02, 0.02, 0.03, 0.04],
    },

    "mat_cfg": "square",
    "mic_array": {
        "tetrahedron": [
            [0.02, 0.0, 0.0],
            [-0.02, 0.0, 0.0],
            [0.0, 0.02, 0.0],
            [0.0, -0.02, 0.0],
        ],
        "square": [
            [0.02, 0.0, 0.0],
            [-0.02, 0.0, 0.0],
            [0.0, 0.02, 0.0],
            [0.0, -0.02, 0.0],
        ],
        "quasi_line": [
            [0.02, 0.0, 0.0],
            [-0.02, 0.0, 0.0],
            [0.0, 0.02, 0.0],
            [0.0, -0.02, 0.0],
        ],
    },

    "perturbations": {
        "mems_pos_std_m": 0.002,  # 2mm standard deviation
        "foa_rot_std_deg": 2.0    # 0 degrees standard deviation
    },

    "num_processes": max(1, cpu_count() - 1),  # Use all but one CPU core
}

vCFG = tCFG.copy()
vCFG["num_rirs"] //= 100

train_output_path = f"train_rir_dataset_{tCFG['mat_cfg']}_{tCFG['sample_rate']}hz_{tCFG['rir_max_length_sec']}s.h5"
valid_output_path = f"valid_rir_dataset_{vCFG['mat_cfg']}_{vCFG['sample_rate']}hz_{vCFG['rir_max_length_sec']}s.h5"

# Utilities

In [3]:
def point_in_polygon(point, polygon):
    x, y = point
    polygon = np.asarray(polygon, dtype=float)

    inside = False
    j = len(polygon) - 1

    for i in range(len(polygon)):
        xi, yi = polygon[i]
        xj, yj = polygon[j]

        crosses_y = (yi > y) != (yj > y)

        if crosses_y:
            x_intersect = (xj - xi) * (y - yi) / (yj - yi) + xi

            if x < x_intersect:
                inside = not inside

        j = i

    return inside


def distance_point_to_segment(point, a, b):
    """
    Distance from point to line segment a-b.
    """
    point = np.asarray(point, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    ab = b - a
    ap = point - a

    denom = np.dot(ab, ab)

    if denom == 0:
        return np.linalg.norm(point - a)

    t = np.dot(ap, ab) / denom
    t = np.clip(t, 0.0, 1.0)

    closest = a + t * ab

    return np.linalg.norm(point - closest)


def distance_to_polygon_edges(point, polygon):
    """
    Minimum distance from point to any polygon edge.
    """
    polygon = np.asarray(polygon, dtype=float)

    min_dist = np.inf

    for i in range(len(polygon)):
        a = polygon[i]
        b = polygon[(i + 1) % len(polygon)]

        dist = distance_point_to_segment(point, a, b)
        min_dist = min(min_dist, dist)

    return min_dist


def random_point_in_polygon(polygon, margin=0.0, max_attempts=1e4):
    polygon = np.asarray(polygon, dtype=float)

    min_x, min_y = polygon.min(axis=0)
    max_x, max_y = polygon.max(axis=0)

    low = [min_x + margin, min_y + margin]
    high = [max_x - margin, max_y - margin]

    if low[0] >= high[0] or low[1] >= high[1]:
        raise ValueError("Margin is too large for this polygon bounding box.")

    for _ in range(int(max_attempts)):
        point = np.random.uniform(low=low, high=high)

        if not point_in_polygon(point, polygon):
            continue

        if margin > 0:
            if distance_to_polygon_edges(point, polygon) < margin:
                continue
        return point

    raise ValueError("Could not find a valid point.")

In [4]:
def random_room(cfg):
    dims = [random.uniform(cfg["min_dim"][i], cfg["max_dim"][i]) for i in range(3)]
    rt60 = random.uniform(*cfg["rt60_range"])
    room_type = random.choice(cfg["types"])
    return dims, rt60, room_type


def random_material(materials):
    key = random.choice(list(materials.keys()))
    coeffs = materials[key]
    return pra.Material({
        "coeffs": coeffs,
        "center_freqs": [125, 250, 500, 1000, 2000]
    }), key


def random_position_in_room(room, margin=0.5):
    polygon = room.floor_corners
    polygon = np.column_stack(polygon)

    x, y = random_point_in_polygon(polygon, margin=margin)

    z = np.random.uniform(
        low=margin,
        high=room.height - margin,
    )

    return [float(x), float(y), float(z)]

In [5]:
def create_room(dims, rt60, fs, material, room_type):
    e_absorption, max_order = pra.inverse_sabine(rt60, dims)

    if room_type == "shoebox":
        max_order = min(max_order, 10)
        room = pra.ShoeBox(
            dims,
            fs=fs,
            materials=material,
            max_order=max_order,
            absorption=e_absorption,
        )
        
        floor_corners = np.array([
            [0.0, dims[0], dims[0], 0.0],
            [0.0, 0.0, dims[1], dims[1]],
        ])
    else:
        num_pts = random.randint(5, 8)
        points = np.random.rand(num_pts, 2) * np.array(dims[:2])
        
        hull = ConvexHull(points)

        # ConvexHull vertices are ordered counter-clockwise
        floor_corners = points[hull.vertices].T

        max_order = min(max_order, 8)
        room = pra.Room.from_corners(
            floor_corners,
            fs=fs,
            materials=material,
            max_order=max_order,
            absorption=e_absorption,
        )
        room.extrude(dims[2])

    room.floor_corners = floor_corners
    room.height = dims[2]

    return room


def add_mems_and_foa(room, center, mic_positions, perturb_cfg):
    mems_pos_std = perturb_cfg["mems_pos_std_m"]
    foa_rot_std = perturb_cfg["foa_rot_std_deg"]

    center = np.array(center).reshape(3, 1)
    mic_positions = np.array(mic_positions).T

    mems_pos = center + mic_positions
    if mems_pos_std > 0:
        # Random noise to coords
        mems_pos += np.random.normal(loc=0.0, scale=mems_pos_std, size=mems_pos.shape)

    foa_pos = np.tile(center, (1, 4))

    all_positions = np.concatenate([mems_pos, foa_pos], axis=1)

    def noisy_angle(base_angle):
        angle = base_angle
        if foa_rot_std > 0:
            angle += np.random.normal(loc=0.0, scale=foa_rot_std)
            if angle < 0:
                angle = -angle
        return angle

    # FOA directions
    dir_x = CardioidFamily(
        orientation=DirectionVector(
            azimuth=noisy_angle(0), 
            colatitude=noisy_angle(90), 
            degrees=True
        ),
        p=0,
    )
    dir_y = CardioidFamily(
        orientation=DirectionVector(
            azimuth=noisy_angle(90), 
            colatitude=noisy_angle(90), 
            degrees=True
        ),
        p=0
    )
    dir_z = CardioidFamily(
        orientation=DirectionVector(
            azimuth=noisy_angle(0), 
            colatitude=noisy_angle(0), 
            degrees=True
        ),
        p=0
    )

    directivities = [
        None, None, None, None,  # MEMS
        None,                    # FOA W
        dir_x, dir_y, dir_z      # FOA X, Y, Z
    ]

    mic_array = pra.MicrophoneArray(all_positions, room.fs, directivity=directivities)
    room.add_microphone_array(mic_array)


def compute_rir(room, source_positions):
    for src_pos in source_positions:
        room.add_source(src_pos)
    room.compute_rir()
    return room.rir

# Visualization

In [6]:
def plot_rir(rir, title="RIR"):
    plt.figure(figsize=(10, 4))
    for ch in range(rir.shape[0]):
        plt.plot(rir[ch], label=f"ch{ch}")
    plt.title(title)
    plt.legend()
    plt.show()

# Generation Loop

In [ ]:
def generate_single_sample(args):
    cfg, seed = args
    
    random.seed(seed)
    np.random.seed(seed % (2**32))
    
    fs = cfg["sample_rate"]
    max_len = int(fs * cfg["rir_max_length_sec"])
    num_sources = cfg["room"]["num_sources"]
    
    # Sometimes, it is impossible to fit every source and microphone in the room, so we re-roll a room if it fails 1e5 amount of times
    while True:
        try:
            # Generate room parameters
            dims, rt60, room_type = random_room(cfg["room"])
            material, mat_name = random_material(cfg["materials"])
            
            # Create room
            room = create_room(dims, rt60, fs, material, room_type)
            
            # Position microphone array
            mic_center = random_position_in_room(room, margin=0.5)
            
            # Generate multiple source positions
            src_positions = [
                random_position_in_room(room, margin=0.1)
                for _ in range(num_sources)
            ]
            
            # Add microphone array to room
            add_mems_and_foa(room, mic_center, cfg["mic_array"][cfg["mat_cfg"]], cfg["perturbations"])
            
            # Compute RIRs
            rir = compute_rir(room, src_positions)

            break
        except:
            print("Incorrect room encountered, re-rolling.")

    def pad_or_crop(arr):
        for mic_idx in range(len(arr)):
            for src_idx in range(len(arr[mic_idx])):
                a = arr[mic_idx][src_idx]
                if a.shape[0] < max_len:
                    a = np.pad(a, (0, max_len - a.shape[0]), mode="constant")
                else:
                    a = a[:max_len]
                arr[mic_idx][src_idx] = a

    # Pad/Crop RIRs
    pad_or_crop(rir)

    rir = np.array(rir)

    # (8, num_sources, max_len) -> (num_sources, 8, max_len)
    rir = np.moveaxis(rir, 0, 1)
    
    mems_rirs = rir[:, :4]
    foa_rirs  = rir[:, 4:]
    
    mic_pos = room.mic_array.R.tolist()
    
    # Return all data needed for this sample
    return {
        "mems_rirs": mems_rirs,  # (num_sources, 4, max_len)
        "foa_rirs": foa_rirs,    # (num_sources, 4, max_len)
        "meta": {
            "room_dim": dims,
            "room_type": room_type,
            "rt60": rt60,
            "material": mat_name,
            "mic_center": mic_center,
            "source_positions": src_positions,
            "actual_mic_positions": mic_pos,
        }
    }


def generate_dataset(cfg, output_path, num_processes=None):
    """Generate RIR dataset using multiprocessing."""
    if num_processes is None:
        num_processes = cfg.get("num_processes", max(1, cpu_count() - 1))
    
    fs = cfg["sample_rate"]
    max_len = int(fs * cfg["rir_max_length_sec"])
    num_sources = cfg["room"]["num_sources"]
    num_rirs = cfg["num_rirs"]
    
    print(f"Generating {num_rirs} RIRs using {num_processes} processes...")
    
    # Prepare arguments for each process
    args_list = [(cfg, i) for i in range(num_rirs)]
    
    # Generate samples
    with Pool(processes=num_processes) as pool:
        results = pool.map(generate_single_sample, args_list)
    
    print("Writing results to HDF5 file...")
    
    # Write results to HDF5 file
    with h5py.File(output_path, "w") as f:
        # Dataset shape: (num_rirs, num_sources, num_channels, max_len)
        mems_ds = f.create_dataset("rir/mems", (num_rirs, num_sources, 4, max_len), dtype="float32")
        foa_ds  = f.create_dataset("rir/foa",  (num_rirs, num_sources, 4, max_len), dtype="float32")
        
        meta = []
        
        # Write results to datasets
        for i, result in enumerate(results):
            mems_ds[i] = result["mems_rirs"]
            foa_ds[i] = result["foa_rirs"]
            meta.append(result["meta"])
        
        f.create_dataset("meta/json", data=str(json.dumps(meta)))
    
    print(f"Dataset saved to {output_path}")

In [ ]:
print("Generating training DS...")
generate_dataset(tCFG, train_output_path)

print("Generating validation DS...")
generate_dataset(vCFG, valid_output_path)

def inspect_sample(path, idx=0, src_idx=0):
    with h5py.File(path, "r") as f:
        mems = f["rir/mems"][idx, src_idx]

    plot_rir(mems, f"MEMS RIR (sample {idx}, source {src_idx})")

inspect_sample(train_output_path, 0, 0)